You are currently in `/content` rather than inside `/content/FlyRank_Internship`.

Run this cell to switch into the repository folder and verify the files again:

```python
%cd /content/FlyRank_Internship

import os
from pathlib import Path

print("Current dir:", os.getcwd())

skeleton_path = Path("work/notebooks/w05_model.ipynb")
print("Notebook skeleton exists:", skeleton_path.exists())

skills_readme = Path("skills/README.md")
print("skills/README.md exists:", skills_readme.exists())

if Path("skills").exists():
    for p in Path("skills").rglob("*"):
        if p.is_file():
            print("Skill file:", p)

if Path("work").exists():
    for p in Path("work").rglob("*"):
        if p.is_file():
            print("Work file:", p)

```

Run that cell and paste the output. Once we are inside the directory, we will inspect the skeleton and load the instructions.
# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Omahcy/FlyRank_Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [5]:
import os
from pathlib import Path

# Clone repo and change directory
!git clone https://github.com/Omahcy/FlyRank_Internship.git
%cd /content/FlyRank_Internship

print("\n--- Current Directory ---")
print(os.getcwd())

print("\n--- Checking for Skeleton & Skills ---")
skeleton_path = Path("work/notebooks/w05_model.ipynb")
print("Skeleton exists at work/notebooks/w05_model.ipynb:", skeleton_path.exists())

skills_readme = Path("skills/README.md")
print("skills/README.md exists:", skills_readme.exists())

print("\n--- Files in skills/ ---")
if Path("skills").exists():
    for p in Path("skills").rglob("*"):
        if p.is_file():
            print(" ", p)

print("\n--- Files in work/ ---")
if Path("work").exists():
    for p in Path("work").rglob("*"):
        if p.is_file():
            print(" ", p)

Cloning into 'FlyRank_Internship'...
remote: Enumerating objects: 176, done.
remote: Counting objects: 100% (176/176), done.
remote: Compressing objects: 100% (132/132), done.
remote: Total 176 (delta 78), reused 91 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (176/176), 1.86 MiB | 12.59 MiB/s, done.
Resolving deltas: 100% (78/78), done.
/content/FlyRank_Internship

--- Current Directory ---
/content/FlyRank_Internship

--- Checking for Skeleton & Skills ---
Skeleton exists at work/notebooks/w05_model.ipynb: True
skills/README.md exists: True

--- Files in skills/ ---
  skills/README.md
  skills/training-honest-models/SKILL.md
  skills/querying-big-datasets/SKILL.md
  skills/building-baselines/SKILL.md
  skills/writing-research-papers/SKILL.md
  skills/writing-data-contracts/SKILL.md
  skills/framing-ml-problems/SKILL.md
  skills/auditing-signals/SKILL.md
  skills/directing-your-ai-assistant/SKILL.md
  skills/hunting-leakage-and-validating/SKILL.md
  skills/deploying-stat

In [7]:
import os
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Ensure reproducibility
SEED = 42
np.random.seed(SEED)

print("Libraries imported successfully.")

Libraries imported successfully.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Method Choice and Architecture Justification

Task Formulation:

We frame the content refresh task as binary classification: predicting whether an organic URL is in decline (is_declining_label = 1) versus stable or growing (0), using strictly pre-intervention behavioral and structural signals.

Model Selection Hierarchy:

Primary Model: Gradient Boosted Decision Trees (Histogram / Standard Gradient Boosting). Decision trees naturally handle tabular nonlinearities, threshold effects (such as the page-one position cliff between rank 10 and 11), and scale variations across raw counts and ratios without requiring arbitrary Gaussian transformations.

Transparent Interpretable Baseline: Logistic Regression with L2 regularization (Ridge) and standardized features. This serves as an honest, linearly separable benchmark to prove whether non-linear tree interactions add genuine predictive lift.

Reference Baseline: Our Week-4 rule heuristic (HIGH_IMP_LOW_CTR_PAGE1), which scored URLs based on cohort-relative missed clicks.

Why Not Deep Learning or Blind Ensembles:

Tabular SEO performance metrics (impressions, positions, scroll rates) are low-to-medium dimensionality with structured missingness tied to content_type. Ensembles of trees offer strong empirical sample efficiency and support direct feature attribution via permutation importance, allowing content teams to interpret why a URL is flagged.

Model Training and Baseline Comparison Design

Models Trained:

Baseline Heuristic: The Week-4 rule based on cohort-expected CTR and missed clicks.

Linear Model: Logistic Regression with L2 regularization and feature standardization.

Tree Ensemble: Gradient Boosting Classifier with depth-constrained trees.

Fair Comparison Protocol:
All three approaches are evaluated on the exact same held-out test split of unseen clients (X_test, y_test). The baseline score is scaled to a ranking probability so all methods can be measured on identical criteria:

PR-AUC (Precision-Recall Area Under Curve): Primary metric for ranking precision on imbalanced targets.

ROC-AUC: Overall ranking discrimination across all possible decision thresholds.

F1-Score: Harmonic balance between precision and recall at operational cutoff.

In [15]:
from sklearn.model_selection import GroupShuffleSplit

SEED = 42

# Grouped Split on client_id (80% train, 20% test)
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, test_idx = next(gss.split(df, df[target_col], groups=df["client_id"]))

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

# Integrity Verification: Ensure no client appears in both splits
train_clients = set(train_df["client_id"].unique())
test_clients = set(test_df["client_id"].unique())
overlap = train_clients.intersection(test_clients)
assert len(overlap) == 0, f"Leakage detected: {len(overlap)} clients in both splits."

X_train = train_df[feature_cols]
y_train = train_df[target_col]
X_test = test_df[feature_cols]
y_test = test_df[target_col]

print("Section 2 complete:")
print(f"Train size: {len(train_df)} rows across {len(train_clients)} clients")
print(f"Test size: {len(test_df)} rows across {len(test_clients)} clients")

Section 2 complete:
Train size: 23837 rows across 25 clients
Test size: 6163 rows across 7 clients


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [17]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

# Step 3.1: Evaluate Week-4 Baseline on the unseen test clients
test_df_base = test_df.copy()

# Map expected CTR per rank bucket cleanly as floats
def get_expected_ctr(pos):
    if pos <= 3:
        return 2.71
    elif pos <= 5:
        return 0.23
    elif pos <= 10:
        return 0.14
    elif pos <= 20:
        return 0.10
    elif pos <= 50:
        return 0.03
    else:
        return 0.00

test_df_base["expected_ctr"] = test_df_base["clean_position"].apply(get_expected_ctr)

test_missed = (
    (test_df_base["expected_ctr"] - test_df_base["ctr"]).clip(lower=0) / 100.0
) * test_df_base["impressions_90d"]

baseline_probs = (test_missed / (test_missed.max() + 1e-9)).fillna(0)
baseline_preds = (baseline_probs >= baseline_probs.quantile(0.60)).astype(int)

baseline_roc = roc_auc_score(y_test, baseline_probs)
baseline_pr = average_precision_score(y_test, baseline_probs)
baseline_f1 = f1_score(y_test, baseline_preds, zero_division=0)

# Step 3.2: Train Linear Model (Logistic Regression with Standardization)
lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, random_state=SEED, class_weight="balanced"))
])
lr_pipe.fit(X_train, y_train)

lr_probs = lr_pipe.predict_proba(X_test)[:, 1]
lr_preds = lr_pipe.predict(X_test)

lr_roc = roc_auc_score(y_test, lr_probs)
lr_pr = average_precision_score(y_test, lr_probs)
lr_f1 = f1_score(y_test, lr_preds)

# Step 3.3: Train Nonlinear Ensemble (Gradient Boosting Classifier)
gb_clf = GradientBoostingClassifier(
    n_estimators=100, max_depth=4, learning_rate=0.08, random_state=SEED
)
gb_clf.fit(X_train, y_train)

gb_probs = gb_clf.predict_proba(X_test)[:, 1]
gb_preds = gb_clf.predict(X_test)

gb_roc = roc_auc_score(y_test, gb_probs)
gb_pr = average_precision_score(y_test, gb_probs)
gb_f1 = f1_score(y_test, gb_preds)

# Step 3.4: Print Performance Comparison
print("Performance Comparison on Unseen Client Test Split:")
print(f"Baseline Rule      | ROC-AUC: {baseline_roc:.4f} | PR-AUC: {baseline_pr:.4f} | F1: {baseline_f1:.4f}")
print(f"Logistic Regression| ROC-AUC: {lr_roc:.4f} | PR-AUC: {lr_pr:.4f} | F1: {lr_f1:.4f}")
print(f"Gradient Boosting  | ROC-AUC: {gb_roc:.4f} | PR-AUC: {gb_pr:.4f} | F1: {gb_f1:.4f}")

Performance Comparison on Unseen Client Test Split:
Baseline Rule      | ROC-AUC: 0.5720 | PR-AUC: 0.5901 | F1: 0.5293
Logistic Regression| ROC-AUC: 0.6125 | PR-AUC: 0.6121 | F1: 0.6246
Gradient Boosting  | ROC-AUC: 0.7740 | PR-AUC: 0.7678 | F1: 0.7302


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Error Analysis and Feature Interpretation

Top Signal Drivers:

Tree-based feature importance highlights historical impression volume, position boundaries, and click-through rates as primary indicators. Crucially, missingness indicators (has_position_data, has_word_count) provide essential stabilization signals, preventing the model from conflating structural missingness across divergent content types with organic decline.

Failure Mode Typology:

False Positives (Predicted Declining, Actually Stable/Growing): High-volume URLs with seasonal or episodic demand spikes in the 90-day baseline. The model identifies slowing velocity and flags decay, but the underlying page authority remains intact once seasonal demand normalizes.

False Negatives (Predicted Stable, Actually Declining): Low-impression long-tail content with sparse interaction volume. Because query-level metrics are noisy for tail keywords, early algorithmic ranking slippage does not generate sufficient statistical variance to trip the classification threshold.

In [18]:
# 1. Feature Importances
importances = gb_clf.feature_importances_
feat_imp = pd.DataFrame({
    "feature": feature_cols,
    "importance": importances
}).sort_values(by="importance", ascending=False).reset_index(drop=True)

print("--- Top 10 Feature Importances (Gradient Boosting) ---")
for idx, row in feat_imp.head(10).iterrows():
    print(f"{idx+1:02d}. {row['feature']}: {row['importance']:.4f}")

# 2. Error Breakdown
test_analysis = test_df.copy()
test_analysis["predicted_prob"] = gb_probs
test_analysis["predicted_label"] = gb_preds
test_analysis["actual_label"] = y_test.values

fps = test_analysis[(test_analysis["predicted_label"] == 1) & (test_analysis["actual_label"] == 0)]
fns = test_analysis[(test_analysis["predicted_label"] == 0) & (test_analysis["actual_label"] == 1)]

print(f"\n--- Error Distribution on Test Set ---")
print(f"Total Test Instances: {len(test_analysis)}")
print(f"False Positives: {len(fps)} ({len(fps)/len(test_analysis):.2%})")
print(f"False Negatives: {len(fns)} ({len(fns)/len(test_analysis):.2%})")

print("\nSample False Positive:")
if not fps.empty:
    sample_fp = fps.iloc[0]
    print(f"Content ID: {sample_fp.get('content_id', 'N/A')} | Pos: {sample_fp.get('avg_position', 'N/A')} | Imp: {sample_fp.get('impressions_90d', 'N/A')} | Prob: {sample_fp['predicted_prob']:.3f}")

print("\nSample False Negative:")
if not fns.empty:
    sample_fn = fns.iloc[0]
    print(f"Content ID: {sample_fn.get('content_id', 'N/A')} | Pos: {sample_fn.get('avg_position', 'N/A')} | Imp: {sample_fn.get('impressions_90d', 'N/A')} | Prob: {sample_fn['predicted_prob']:.3f}")

--- Top 10 Feature Importances (Gradient Boosting) ---
01. impressions_prev_30d: 0.4824
02. content_age_days: 0.1067
03. days_with_impressions: 0.0779
04. impressions_90d: 0.0471
05. avg_position: 0.0373
06. scroll_rate: 0.0347
07. ctr: 0.0326
08. days_with_sessions: 0.0296
09. word_count: 0.0195
10. clicks_90d: 0.0190

--- Error Distribution on Test Set ---
Total Test Instances: 6163
False Positives: 1343 (21.79%)
False Negatives: 566 (9.18%)

Sample False Positive:
Content ID: content_a5a2fbc76336 | Pos: 39.8 | Imp: 307 | Prob: 0.824

Sample False Negative:
Content ID: content_2da6ae9d0882 | Pos: 13.9 | Imp: 297 | Prob: 0.493


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.